# 🍽️ Weekly Themed Meal Planner MVP

A meal planning assistant that generates 7 themed dinners with consolidated shopping lists.

## Features
- 7 unique themed dinners (Monday-Sunday)
- 5 ingredients per recipe (excluding basics)
- Dietary preference support (vegan, vegetarian, keto, omnivore)
- Consolidated shopping list
- Email delivery

## 1. Installation & Setup

In [1]:
%pip install -q langchain langchain-openai pydantic gradio python-dotenv

Note: you may need to restart the kernel to use updated packages.


## 2. Configuration

Create a file called `.env` in the same folder as this notebook with this line:

```
OPENAI_API_KEY=sk-...
```

**Edit the values below before running the notebook.**

In [2]:
import os
from dotenv import load_dotenv

load_dotenv('.env')
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', '')
OPENAI_MODEL = os.getenv('OPENAI_MODEL', 'gpt-4o-mini')
TEMPERATURE = 0.7
EMAIL_SENDER = os.getenv('EMAIL_SENDER', '')
EMAIL_PASSWORD = os.getenv('EMAIL_PASSWORD', '')
INGREDIENTS_PER_RECIPE = 5
EXCLUDED_BASICS = ['salt', 'pepper', 'oil', 'garlic', 'water']
DAYS = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# Never hardcode credentials in notebooks or Git history.
# Missing credentials are reported only when the corresponding feature is used.

In [3]:
# Configuration verification (never print the actual key).
print('OpenAI configured:', bool(OPENAI_API_KEY))
print('Email configured:', bool(EMAIL_SENDER and EMAIL_PASSWORD))

OpenAI configured: True
Email configured: False


## 3. Pydantic Models for Validation

In [4]:
from pydantic import BaseModel, Field, field_validator
from typing import List, Optional
from enum import Enum


class DietaryPreference(str, Enum):
    OMNIVORE = "omnivore"
    VEGETARIAN = "vegetarian"
    VEGAN = "vegan"
    KETO = "keto"


class Ingredient(BaseModel):
    """Single ingredient with quantity."""
    name: str = Field(..., description="Name of the ingredient")
    quantity: str = Field(..., description="Quantity with unit (e.g., '2 cups', '500g')")


class Recipe(BaseModel):
    """A single themed dinner recipe."""
    day: str = Field(..., description="Day of the week")
    theme: str = Field(..., description="Cuisine theme (e.g., 'Italian', 'Mexican')")
    dish_name: str = Field(..., description="Name of the dish")
    ingredients: List[Ingredient] = Field(..., description="List of 5 ingredients")
    instructions: str = Field(..., description="Brief cooking instructions")

    @field_validator('ingredients')
    @classmethod
    def validate_ingredient_count(cls, v):
        if len(v) != INGREDIENTS_PER_RECIPE:
            raise ValueError(f"Recipe must have exactly {INGREDIENTS_PER_RECIPE} ingredients, got {len(v)}")
        return v


class ShoppingItem(BaseModel):
    """Consolidated shopping list item."""
    name: str = Field(..., description="Ingredient name")
    total_quantity: str = Field(..., description="Total quantity needed")
    used_in: List[str] = Field(..., description="List of dishes using this ingredient")


class MealPlan(BaseModel):
    """Complete weekly meal plan."""
    dietary_preference: str = Field(..., description="Selected dietary preference")
    recipes: List[Recipe] = Field(..., description="List of 7 recipes for the week")
    shopping_list: List[ShoppingItem] = Field(..., description="Consolidated shopping list")

    @field_validator('recipes')
    @classmethod
    def validate_recipe_count(cls, v):
        if len(v) != len(DAYS):
            raise ValueError(f"Meal plan must have exactly {len(DAYS)} recipes, got {len(v)}")
        return v

    @field_validator('recipes')
    @classmethod
    def validate_unique_themes(cls, v):
        themes = [r.theme.lower() for r in v]
        if len(themes) != len(set(themes)):
            raise ValueError("All themes must be unique")
        return v

## 4. LangChain Setup & Prompts

In [5]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser


def get_llm():
    """Get the configured LLM."""
    if not OPENAI_API_KEY:
        raise ValueError('get_llm: OPENAI_API_KEY is missing. Add it to .env.')
    return ChatOpenAI(model=OPENAI_MODEL, temperature=TEMPERATURE, api_key=OPENAI_API_KEY)


# Main meal plan generation prompt
MEAL_PLAN_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a professional meal planner. Generate a weekly meal plan with EXACTLY 7 themed dinners.

STRICT RULES - FOLLOW EXACTLY:
1. Each recipe MUST have EXACTLY 5 main ingredients (do NOT count salt, pepper, oil, garlic, or water)
2. ALL 7 THEMES MUST BE COMPLETELY DIFFERENT - no duplicates allowed! Use: Italian, Mexican, Japanese, Indian, Thai, Greek, American, Chinese, French, Korean, Mediterranean, Middle Eastern, etc.
3. All ingredients MUST be real, commonly available items
4. Follow the dietary preference strictly
5. Provide realistic quantities for 2 servings

Dietary Guidelines:
- Omnivore: Any ingredients allowed
- Vegetarian: No meat or fish, eggs and dairy OK
- Vegan: No animal products at all
- Keto: Low carb, high fat, no grains/sugar/starchy vegetables

Output valid JSON matching this exact structure:
{format_instructions}"""),
    ("human", """Create a weekly meal plan:
- Diet: {dietary_preference}
- Theme requests: {theme_requests}
- Preferences: {additional_preferences}

IMPORTANT: Each day MUST have a DIFFERENT cuisine theme. Generate 7 unique themed dinners for Monday-Sunday.""")
])


# Shopping list consolidation prompt
SHOPPING_LIST_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a shopping list organizer. Consolidate ingredients from multiple recipes into a single shopping list.

RULES:
1. Combine same ingredients and sum their quantities
2. List which dishes use each ingredient
3. Use standard units (cups, tablespoons, grams, pieces)
4. Round up quantities for convenience

Output valid JSON matching this structure:
{format_instructions}"""),
    ("human", """Consolidate the shopping list from these recipes:
{recipes_json}""")
])

## 5. Meal Generation Logic

In [6]:
import json
import time
import re
from pydantic import ValidationError
from openai import AuthenticationError, RateLimitError, APIConnectionError, APITimeoutError, APIStatusError


def error_result(function, exc, suggestion):
    """Provide consistent, actionable errors without exposing credentials."""
    return {'success': False, 'meal_plan': None, 'generation_time': None,
            'error': f'{function}: {type(exc).__name__}: {exc}. Suggestion: {suggestion}'}


def normalize_preference(value, default):
    """Safely normalize blank form inputs, including None."""
    return value.strip() if isinstance(value, str) and value.strip() else default


def ingredient_text(ingredients):
    """Share ingredient formatting between UI and email."""
    return ', '.join(f"{item['quantity']} {item['name']}" for item in ingredients)


def parse_meal_plan(content):
    """Extract and validate a JSON object from an LLM response."""
    if not isinstance(content, str):
        raise TypeError('Model response must be text')
    cleaned = content.strip()
    match = re.fullmatch(r'```(?:json)?\s*(.*?)\s*```', cleaned, flags=re.DOTALL | re.IGNORECASE)
    if match:
        cleaned = match.group(1)
    data = json.loads(cleaned)
    return MealPlan.model_validate(data).model_dump()


class MealPlanGenerator:
    """Orchestrate the model call and validation, with injectable LLM for tests."""
    def __init__(self, llm=None):
        self.llm = llm
        self.recipe_parser = JsonOutputParser(pydantic_object=MealPlan)

    def call_model(self, dietary_preference, theme_requests, additional_preferences):
        """Perform the remote API request only."""
        model = self.llm if self.llm is not None else get_llm()
        chain = MEAL_PLAN_PROMPT | model
        return chain.invoke({
            'dietary_preference': dietary_preference,
            'theme_requests': normalize_preference(theme_requests, 'random diverse cuisines'),
            'additional_preferences': normalize_preference(additional_preferences, 'none'),
            'format_instructions': self.recipe_parser.get_format_instructions()
        })

    def generate(self, dietary_preference, theme_requests='random', additional_preferences='none') -> dict:
        """Coordinate one generation attempt; return a consistent result schema."""
        start = time.time()
        try:
            if dietary_preference not in [pref.value for pref in DietaryPreference]:
                raise ValueError(f'Unsupported diet: {dietary_preference!r}')
            response = self.call_model(dietary_preference, theme_requests, additional_preferences)
            plan = parse_meal_plan(response.content)
            return {'success': True, 'meal_plan': plan,
                    'generation_time': round(time.time() - start, 2), 'error': None}
        except json.JSONDecodeError as exc:
            return error_result('parse_meal_plan', exc, f'Check JSON syntax at line {exc.lineno}, column {exc.colno}')
        except ValidationError as exc:
            return error_result('parse_meal_plan', exc, 'Check recipe count, ingredient count and unique themes')
        except (AuthenticationError, RateLimitError, APIConnectionError, APITimeoutError, APIStatusError) as exc:
            return error_result('call_model', exc, 'Check API credentials, network, rate limits or service status')
        except (ValueError, TypeError, KeyError, AttributeError) as exc:
            return error_result('generate', exc, 'Check input values, model configuration and response format')
        except Exception as exc:
            return error_result('generate', exc, 'Inspect the exception and application logs')


generator = MealPlanGenerator()

## 6. Email Functionality

In [7]:
import smtplib
from email.message import EmailMessage
from email.utils import parseaddr


def format_meal_plan_email(meal_plan: dict) -> str:
    """Format shopping list first, then the seven recipes."""
    lines = ['=' * 40, f"WEEKLY MEAL PLAN ({meal_plan['dietary_preference'].upper()})", '=' * 40, '',
             'SHOPPING LIST', '-' * 20]
    for item in meal_plan['shopping_list']:
        lines.append(f"- {item['total_quantity']} {item['name']}")
    lines += ['', '=' * 40, 'RECIPES', '=' * 40]
    for recipe in meal_plan['recipes']:
        lines.extend([f"\n{recipe['day'].upper()} - {recipe['theme']}", recipe['dish_name'],
                      f"Ingredients: {ingredient_text(recipe['ingredients'])}",
                      f"Instructions: {recipe['instructions']}"])
    return '\n'.join(lines)


def build_email(recipient, meal_plan):
    """Build a message without contacting a server."""
    if not recipient or parseaddr(recipient)[1] != recipient or '@' not in recipient:
        raise ValueError('Invalid recipient email address')
    if not EMAIL_SENDER:
        raise ValueError('EMAIL_SENDER is missing from .env')
    msg = EmailMessage()
    msg['From'] = EMAIL_SENDER
    msg['To'] = recipient
    msg['Subject'] = 'Your Weekly Meal Plan'
    msg.set_content(format_meal_plan_email(meal_plan))
    return msg


def send_meal_plan_email(recipient_email: str, meal_plan: dict) -> dict:
    """Send a previously generated plan, never regenerate it."""
    try:
        if not EMAIL_PASSWORD:
            raise ValueError('EMAIL_PASSWORD is missing from .env')
        msg = build_email(recipient_email, meal_plan)
        with smtplib.SMTP('smtp.gmail.com', 587, timeout=20) as server:
            server.starttls()
            server.login(EMAIL_SENDER, EMAIL_PASSWORD)
            server.send_message(msg)
        return {'success': True, 'message': f'Email sent to {recipient_email}'}
    except (ValueError, smtplib.SMTPException, OSError) as exc:
        return {'success': False, 'message': f'send_meal_plan_email: {type(exc).__name__}: {exc}. Suggestion: check recipient, SMTP settings and credentials.'}

## 7. Validation

In [8]:
def run_basic_validation(meal_plan_result: dict) -> dict:
    """Check a generated plan without making API calls."""
    if not meal_plan_result.get('success'):
        return {'passed': False, 'errors': [meal_plan_result.get('error')]}
    try:
        MealPlan.model_validate(meal_plan_result['meal_plan'])
        return {'passed': True, 'errors': []}
    except (ValidationError, KeyError, TypeError) as exc:
        return {'passed': False, 'errors': [f'run_basic_validation: {type(exc).__name__}: {exc}']}

## 8. Gradio Interface

In [9]:
import gradio as gr
import time as pytime

# Gradio State preserves the exact plan displayed when the user clicks Email Plan.
def format_shopping_list(meal_plan: dict) -> str:
    lines = ['## Shopping List\n']
    for item in meal_plan['shopping_list']:
        lines.append(f"- {item['total_quantity']} **{item['name']}**")
    return '\n'.join(lines)


def format_single_recipe(recipe: dict) -> str:
    return (f"**{recipe['day']}** | {recipe['theme']} | *{recipe['dish_name']}*\n"
            f"{ingredient_text(recipe['ingredients'])}\n")


def generate_plan_streaming(dietary_pref, theme_requests, additional_prefs):
    """Stream display updates while preserving the generated plan in state."""
    yield 'Generating your meal plan...', None
    result = generator.generate(dietary_pref, theme_requests, additional_prefs)
    if not result['success']:
        yield f"Error: {result['error']}", None
        return
    plan = result['meal_plan']
    output = (f"## Weekly {plan['dietary_preference'].title()} Meal Plan\n\n"
              + format_shopping_list(plan) + '\n\n---\n\n## Recipes\n\n')
    yield output, plan
    for recipe in plan['recipes']:
        output += format_single_recipe(recipe) + '\n'
        yield output, plan
    validation = run_basic_validation(result)
    status = 'All checks passed' if validation['passed'] else f"Issues: {', '.join(validation['errors'])}"
    yield output + f"---\n*Generated in {result['generation_time']}s | {status}*", plan


def send_email_ui(email, saved_plan):
    """Email exactly the plan currently displayed, without another API call."""
    if not saved_plan:
        return 'Generate a plan before emailing it.'
    return send_meal_plan_email(email, saved_plan)['message']


with gr.Blocks(title='Weekly Meal Planner') as app:
    gr.Markdown('# Weekly Meal Planner\nGenerate 7 themed dinners with a consolidated shopping list.')
    current_plan = gr.State(value=None)
    with gr.Row():
        with gr.Column(scale=1):
            dietary_dropdown = gr.Dropdown(choices=['omnivore', 'vegetarian', 'vegan', 'keto'], value='omnivore', label='Diet')
            theme_input = gr.Textbox(label='Themes (optional)', placeholder='Italian, Mexican...', lines=1)
            additional_input = gr.Textbox(label='Preferences (optional)', placeholder='no nuts, quick meals...', lines=1)
            generate_btn = gr.Button('Generate', variant='primary')
            gr.Markdown('---')
            email_input = gr.Textbox(label='Email', placeholder='you@example.com')
            send_btn = gr.Button('Email Plan')
            email_status = gr.Textbox(label='Status', interactive=False)
        with gr.Column(scale=2):
            output_display = gr.Markdown('Click **Generate** to start.')
    generate_btn.click(fn=generate_plan_streaming,
                       inputs=[dietary_dropdown, theme_input, additional_input],
                       outputs=[output_display, current_plan])
    send_btn.click(fn=send_email_ui, inputs=[email_input, current_plan], outputs=[email_status])

/opt/anaconda3/envs/bootcamp-env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
app.launch(inbrowser=True, share=False)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [15]:
try:
    parse_meal_plan('{"recipes": [}')
except Exception as e:
    print("Function: parse_meal_plan")
    print("Error type:", type(e).__name__)
    print("Message:", str(e))
    if isinstance(e, json.JSONDecodeError):
        print("Line:", e.lineno)
        print("Column:", e.colno)
        print("Suggestion: Check the JSON syntax.")

Function: parse_meal_plan
Error type: JSONDecodeError
Message: Expecting value: line 1 column 14 (char 13)
Line: 1
Column: 14
Suggestion: Check the JSON syntax.


In [16]:
try:
    with open("nonexistent_test_file.json", "r") as file:
        data = json.load(file)

except FileNotFoundError as e:
    print("Function: File loading test")
    print("Error type:", type(e).__name__)
    print("Message:", str(e))
    print("Suggestion: Check that the file exists and the path is correct.")
    

Function: File loading test
Error type: FileNotFoundError
Message: [Errno 2] No such file or directory: 'nonexistent_test_file.json'
Suggestion: Check that the file exists and the path is correct.


In [17]:
from pydantic import ValidationError

try:
    MealPlan.model_validate({"recipes": []})

except ValidationError as e:
    print("Function: MealPlan.model_validate")
    print("Error type:", type(e).__name__)
    print("Validation errors:")
    for error in e.errors():
        print("Field:", error["loc"])
        print("Message:", error["msg"])
    print("Suggestion: Check the required recipe fields and number of recipes.")

Function: MealPlan.model_validate
Error type: ValidationError
Validation errors:
Field: ('dietary_preference',)
Message: Field required
Field: ('recipes',)
Message: Value error, Meal plan must have exactly 7 recipes, got 0
Field: ('shopping_list',)
Message: Field required
Suggestion: Check the required recipe fields and number of recipes.


## 9. Launch Application

In [11]:
# Start the interface locally. Do not create a public share link by default.
# Uncomment the next line when you want to use the UI:
# app.launch(share=False)

## 10. Run Test Cases

Runs every test case through the generator and the basic validation checks. This makes 4 real API calls.

In [12]:
# Live integration tests: intentionally opt-in because they use paid API requests.
TEST_CASES = [
    {'dietary_preference': 'omnivore', 'theme_requests': 'random', 'additional_preferences': 'none'},
    {'dietary_preference': 'vegetarian', 'theme_requests': 'Italian, Asian, Mexican', 'additional_preferences': 'no mushrooms'},
    {'dietary_preference': 'vegan', 'theme_requests': 'random', 'additional_preferences': 'high protein'},
    {'dietary_preference': 'keto', 'theme_requests': 'Mediterranean, American', 'additional_preferences': 'none'}
]
RUN_LIVE_API_TESTS = False
if RUN_LIVE_API_TESTS:
    for case in TEST_CASES:
        result = generator.generate(**case)
        validation = run_basic_validation(result)
        print(case['dietary_preference'], '->', 'PASS' if validation['passed'] else validation['errors'])
else:
    print('Live API tests skipped. Set RUN_LIVE_API_TESTS=True after configuring .env.')

Live API tests skipped. Set RUN_LIVE_API_TESTS=True after configuring .env.


## 11. Offline regression and error-handling tests
These tests do not use OpenAI or SMTP credentials. They verify parsing, schema checks, missing files, and error responses.

In [13]:
from pathlib import Path
from tempfile import TemporaryDirectory
from types import SimpleNamespace


def load_test_cases(path):
    """Load test inputs from JSON, with actionable file errors."""
    try:
        return json.loads(Path(path).read_text(encoding='utf-8'))
    except FileNotFoundError as exc:
        raise FileNotFoundError(f'load_test_cases: {exc}. Suggestion: check file path {path}') from exc
    except json.JSONDecodeError as exc:
        raise ValueError(f'load_test_cases: JSONDecodeError at line {exc.lineno}, column {exc.colno}. Suggestion: fix JSON syntax') from exc


def sample_plan():
    recipes = []
    for day, theme in zip(DAYS, ['Italian', 'Mexican', 'Japanese', 'Indian', 'Thai', 'Greek', 'Korean']):
        recipes.append({'day': day, 'theme': theme, 'dish_name': f'{theme} dinner',
                        'ingredients': [{'name': f'ingredient{i}', 'quantity': '1 cup'} for i in range(5)],
                        'instructions': 'Cook and serve.'})
    return {'dietary_preference': 'vegetarian', 'recipes': recipes,
            'shopping_list': [{'name': 'ingredient0', 'total_quantity': '7 cups', 'used_in': ['dinner']} ]}


def assert_raises(expected, action):
    try:
        action()
    except expected as exc:
        return f'PASS ({type(exc).__name__}: {str(exc)[:110]})'
    raise AssertionError(f'Expected {expected.__name__}')


plan = sample_plan()
assert len(parse_meal_plan(json.dumps(plan))['recipes']) == len(DAYS)
assert len(parse_meal_plan('```json\n' + json.dumps(plan) + '\n```')['recipes']) == len(DAYS)
print('Valid JSON and fenced JSON: PASS')
print('Malformed JSON:', assert_raises(json.JSONDecodeError, lambda: parse_meal_plan('{"recipes": [}')))
invalid = sample_plan()
invalid['recipes'][0]['ingredients'] = invalid['recipes'][0]['ingredients'][:4]
print('Invalid schema:', assert_raises(ValidationError, lambda: parse_meal_plan(json.dumps(invalid))))
print('Missing file:', assert_raises(FileNotFoundError, lambda: load_test_cases('does_not_exist.json')))
with TemporaryDirectory() as folder:
    path = Path(folder) / 'malformed.json'
    path.write_text('{bad json', encoding='utf-8')
    print('Invalid file JSON:', assert_raises(ValueError, lambda: load_test_cases(path)))
assert normalize_preference(None, 'none') == 'none'
assert ingredient_text(plan['recipes'][0]['ingredients']).startswith('1 cup')
print('Helper functions: PASS')

class FailingGenerator(MealPlanGenerator):
    def call_model(self, *args):
        raise ConnectionError('Simulated offline connection failure')

failure = FailingGenerator().generate('vegetarian')
assert failure['success'] is False and 'ConnectionError' in failure['error']
assert set(failure) == {'success', 'meal_plan', 'generation_time', 'error'}
print('Simulated API/network failure:', failure['error'])

class FakeGenerator(MealPlanGenerator):
    def call_model(self, *args):
        return SimpleNamespace(content=json.dumps(plan))

success = FakeGenerator().generate('vegetarian')
assert success['success'] and run_basic_validation(success)['passed']
assert set(success) == set(failure)
print('Fake model generation and validation: PASS')
assert send_email_ui('test@example.com', None) == 'Generate a plan before emailing it.'
print('Email UI without generated plan: PASS')
print('ALL OFFLINE TESTS PASSED')

Valid JSON and fenced JSON: PASS
Malformed JSON: PASS (JSONDecodeError: Expecting value: line 1 column 14 (char 13))
Invalid schema: PASS (ValidationError: 1 validation error for MealPlan
recipes.0.ingredients
  Value error, Recipe must have exactly 5 ingredients, g)
Missing file: PASS (FileNotFoundError: load_test_cases: [Errno 2] No such file or directory: 'does_not_exist.json'. Suggestion: check file path does_)
Invalid file JSON: PASS (ValueError: load_test_cases: JSONDecodeError at line 1, column 2. Suggestion: fix JSON syntax)
Helper functions: PASS
Simulated API/network failure: generate: ConnectionError: Simulated offline connection failure. Suggestion: Inspect the exception and application logs
Fake model generation and validation: PASS
Email UI without generated plan: PASS
ALL OFFLINE TESTS PASSED


## 12. Before / after comparison

| Original | Refactored |
|---|---|
| API invocation outside `try` | API invocation protected by error handling |
| JSON parsing embedded in `generate()` | `parse_meal_plan()` is independently testable |
| Hardcoded email placeholders | Credentials read from `.env` |
| Inconsistent success/failure return keys | Consistent result keys in both cases |
| Duplicate ingredient formatting | `ingredient_text()` shared helper |
| Email button regenerates a different plan | Gradio `State` holds the displayed plan |
| Repeated Pydantic rules | Pydantic model is the single validation source |
| Automatic four paid API test calls | Opt-in live tests plus offline regression tests |
| Public `share=True` launch | Local-only launch, explicitly opt-in |

**Limitations:** Real API success and SMTP delivery must be verified with valid credentials in the user environment. The offline tests do not establish either.